<a href="https://colab.research.google.com/github/gph05010/Daily-Study-Log-DeepLearning/blob/main/ex10_Transformer(Bert%2C_Kobert%2C_KoElctra%2C_KoBart)_%EB%84%A4%EC%9D%B4%EB%B2%84%EC%98%81%ED%99%94%EB%A6%AC%EB%B7%B0%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 학습목표
  - 영화리뷰 데이터 감성분류
    - Transformer, Bert 기반의 모델 활용 추가학습

# 3가지 타입의 한국어 언어모델
- Encoder 중심의 모델 : BERT 계열

<table>
<tr align="center">
  <td width="50"><b>모델명</b>
  <td width="50"><b>개발자</b>
  <td width="100"><b>학습데이터</b>
  <td width="100"><b>Tokenizer</b>
  <td width="100"><b>단어수</b>
  <td width="50"><b>파라미터수</b>
<tr align="center">
  <td>KorBERT
  <td>ETRI
  <td>뉴스/백과사전<br>23GB
  <td>Mophologpy,<br>WordPiece
  <td>30,349(Mophologpy),<br>30,797(WordPiece)
  <td>110M
<tr align="center">
  <td>KoBERT
  <td>SKT
  <td>위키피디아<br>20M
  <td>Sentencce-Piece
  <td>8,002
  <td>92M
<tr align="center">
  <td>HanBERT
  <td>투블럭AI
  <td>일반/특허문서<br>70GB
  <td>Moran
  <td>54,000
  <td>128M
<tr align="center">
  <td>KoreALBERT
  <td>삼성SDS
  <td>위키피디아/나무위키/뉴스<br>책 줄거리 요약 등<br>43GB
  <td>Sentencce-Piece
  <td>32,000
  <td>12M, 18M
<tr align="center">
  <td>KLUE-BERT
  <td>Klue project
  <td>모두의말뭉치/CC-100-Kor/<br>나무위키/뉴스/청원 등<br>63GB
  <td>Morpheme-based<br>subword
  <td>32,000
  <td>111M
<tr align="center">
  <td>KRBERT
  <td>서울대
  <td>위키피디아/뉴스
  <td>WordPiece
  <td>16,424(Character)<br>12,367(Subcharacter)
  <td>99M(Character)<br>96M(Subcharacter)
<tr align="center">
  <td>DistillKoBERT
  <td>개인(박장원)
  <td>위키피디아/나무위키/<br>뉴스 등
  <td>Sentence-Piece
  <td>30,522
  <td>27.8M
<tr align="center">
  <td>KcBERT
  <td>개인(이준범)
  <td>네이버뉴스의 댓글/대댓글
  <td>Word-Piece
  <td>30,000
  <td>109M
<tr align="center">
  <td>KcELETRA
  <td>개인(이준범)
  <td>네이버뉴스의 댓글/대댓글
  <td>Word-Piece
  <td>30,000
  <td>124M
<tr align="center">
  <td>KoBigBird
  <td>개인(박장원)
  <td>위키피디아/뉴스/모두의<br>말뭉치/Common Crawl 등
  <td>Word-Piece
  <td>32,500
  <td>113.8M                   
</table>


- 영화 리뷰데이터 불러오기
  - https://github.com/e9t/nsmc/
  - ratings_train.txt, ratings_test.txt 파일 다운로드

In [1]:
%cd /content/drive/MyDrive/00 딥러닝

/content/drive/MyDrive/00 딥러닝


In [2]:
!pwd

/content/drive/MyDrive/00 딥러닝


In [5]:
import pandas as pd
train_data = pd.read_csv('./data/ratings_train.txt', delimiter = '\t')
test_data = pd.read_csv('./data/ratings_test.txt', delimiter = '\t')

- 각 파일은 id, document, label 세 개의 컬럼으로 구성
  - id : 리뷰아이디
  - document : 실제 리뷰
  - label : 리뷰의 감정 클래스(0 : 부정, 1 : 긍정)

In [6]:
train_data.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [7]:
train_data.info() # 결측치 5개 발견

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [8]:
test_data.info() # 결측치 3개 발견

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        50000 non-null  int64 
 1   document  49997 non-null  object
 2   label     50000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.1+ MB


In [9]:
# 결측치 제거
train_data.dropna(inplace=True)
test_data.dropna(inplace=True)

In [10]:
train_data.shape, test_data.shape

((149995, 3), (49997, 3))

### 전처리
- 한글 띄어쓰기만 두고 나머지는 삭제
- 공백만 남아있는 리뷰 삭제
### 정규표현식을 활용한 전처리
- 규칙
  - ^ : 부정(NOT)
  - 가-힣 : 한글 전체범위
  - \s : 공백(space, tab)
- 기능
  - replace("바꾸기전", "바꾼후", regex = True)
  - regex = True -> 정규표현식을 기반으로 치환

In [15]:
# 한글과 공백을 제외한 나머지는 전부 제거(영어, 숫자, 특수문자 등등)
train_data['document'] = train_data['document'].str.replace(r'[^가-힣\s]', '', regex=True) # str : 문자열을 가져오기, replace 함수는 문자열을 사용하는 함수
test_data['document'] = test_data['document'].str.replace(r'[^가-힣\s]', '', regex=True) # str : 문자열을 가져오기, replace 함수는 문자열을 사용하는 함수

In [16]:
train_data.head()

,id,document,label
0,9976970,아 더빙 진짜 짜증나네요 목소리,0
1,3819312,흠포스터보고 초딩영화줄오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 솔직히 재미는 없다평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화스파이더맨에서 늙어보이기만 했던 커스틴 던...,1


- 전처리2
  - 한국어 특성상 단순 띄어쓰기 기준으로 토큰화를 진행할 경우 정확도가 떨어진다(조사, 어미 등등)
  - Bert 모델의 전용 토큰화 도구를 사용해야 함(띄어쓰기 기준)
  - 한글 리뷰를 형태소 분석 후에 다시 문장으로 만들어 Bert Tokenizer에 입력해주는 과정이 필요하다

## 한국어 토큰화 도구 비교
- https://konlpy.org/ko/v0.6.0/morph/

| **토큰화 도구** | **설명** | **장점** | **단점** | **사용하면 좋은 글 유형** |
|----------------|---------|---------|---------|------------------|
| **KoNLPy - Okt** | 트위터에서 개발한 한국어 형태소 분석기 | - 속도가 빠름 <br> - 줄임말, 신조어에 강함 <br> - 명사, 동사, 형용사 태깅 지원 | - 분석 정확도가 상대적으로 낮음 <br> - 긴 문장에서 오탈자나 오류 발생 가능 | - SNS 댓글, 트위터, 카톡 대화 등 구어체 데이터 |
| **KoNLPy - Mecab** | 일본 Mecab을 한국어에 맞게 수정한 형태소 분석기 | - 속도가 가장 빠름 <br> - 비교적 높은 정확도 <br> - 메모리 사용량 적음 | - 윈도우 환경에서 설치가 까다로움 <br> - 사용자 사전 추가 필요 | - 뉴스, 문서 요약, 법률 문서 등 정형 데이터 |
| **KoNLPy - Kkma** | 서울대에서 개발한 형태소 분석기 | - 구문 분석 기능 포함 <br> - 문장 분리 기능 제공 | - 속도가 느림 <br> - 명사 추출 시 과도한 분할 발생 가능 | - 학술 논문, 문서 분석, 보고서 |
| **KoNLPy - Komoran** | Shineware에서 개발한 형태소 분석기 | - 딥러닝 기반으로 정확도 높음 <br> - 사용자 사전 확장 가능 | - 속도가 느린 편 | - 뉴스 기사, 공식 문서, 챗봇 |
| **KoNLPy - Hannanum** | KAIST에서 개발한 한국어 형태소 분석기 | - 어절 분석 및 띄어쓰기 보정 기능 제공 <br> - 구문 분석 가능 | - 속도가 느림 <br> - 비교적 오래된 기술로 최신 트렌드 반영 부족 | - 논문, 신문 기사, 공식 문서 |
| **Kiwi** | 한국어 형태소 분석기 (KyTea 기반) | - 최신 기술 적용 <br> - 속도와 정확도 균형 잡힘 <br> - 신조어, 띄어쓰기 보정 기능 제공 | - 비교적 새로운 도구라 참고 자료 부족 | - 검색 엔진, 챗봇, 감성 분석 |


In [17]:
# 형태소 분석 -> konlpy 도구 설치 -> Okt (리뷰, sns, 댓글과 같이 인터넷 용어 및 신조어에 강하다)
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.0/438.0 kB 30.9 MB/s eta 0:00:00


In [18]:
import konlpy
from konlpy.tag import Okt
from tqdm import tqdm # 로딩 바를 활용하여 진행 상황을 표기해주는 도구

In [23]:
# tqdm을 pandas와 연결하여 사용
tqdm.pandas()

# 형태소 분석기 객체 생성
okt = Okt()

# 토큰화 수행 -> 어간 추출(stemming)
# lambda 함수를 적용하여 전체 데이터 토큰화 수행 -> progress_map() 활용
train_data['document'] = train_data['document'].progress_map(
    # 형태소 분석 후 띄어쓰기를 기준으로 다시 합쳐주는 함수 생성
    lambda x : ' '.join(okt.morphs(x, stem = True, norm = True)) # norm : normalization(정규화)
)

100%|██████████| 149995/149995 [08:50<00:00, 282.93it/s]


In [22]:
' '.join(['안녕', '하세요'])

'안녕 하세요'

In [24]:
# 전처리된 데이터를 구글드라이브에 저장
import pickle
# pickle: save + load 기능을 담당하는 파이썬용 저장기
# DataFrame, list, dict ,model 등 파이썬용 객체들을 그대로 저장 -> 타입 구조를 보존

with open('./data/bert_train_data_ratings.pkl', 'wb') as f:
  pickle.dump(train_data, f)